In [1]:
import PyPDF2
import pandas as pd
import re
from PyPDF2 import PdfReader

def extract_pdf_data(file_path):
    unidades_medida = ["MCG", "%", "UI", "MEQ", "MG", "ML", "UI/ML", "MCG/ML", "%/ML", "G/ML", "ML/ML", "MUI/ML"]

    reader = PdfReader(file_path)
    text = ''
    for page in reader.pages:
        text += page.extract_text()
    
    # Regex patterns to find the start of each section and the end of the table
    sections = re.split(r'(Código:)', text)
    
    # Initializing lists to store data
    cotacoes = []
    
    for i in range(1, len(sections), 2):
        section_text = sections[i] + sections[i+1]
        
        # Extraindo informações do Pedido
        numero_pedido_match = re.search(r'Data Entrega:\s*(\d+\.\d+)', section_text)
        cnpj_filial_match = re.search(r'CNPJ:\s*(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})', section_text)
        data_emissao_match = re.search(r'(\d{2}/\d{2}/\d{4})', section_text)
        condicao_pagamento_match = re.search(r'\n(.*?)DIASCotação:', section_text)
        
        numero_pedido = numero_pedido_match.group(1)
        cnpj_filial = cnpj_filial_match.group(1)
        data_emissao = data_emissao_match.group(1)
        condicao_pagamento = condicao_pagamento_match.group(1)

        #print(section_text)
        
        # Extraindo Tabela de Itens
        table_start = section_text.find('Produto Qtd. Desc. (%) Total Preço Compra Cód. Barras Fabricante')
        table_end = section_text.find('Totais', table_start)

        
        if table_start == -1 or table_end == -1:
            continue
        
        table_text = section_text[table_start:table_end]

        #print(table_text)
        
        # Processar a tabela de itens
        items = []
        rows = table_text.split('\n')[1:]  # Ignorar o cabeçalho
        #print(rows)
        current_item = []
        qtd = []
        current_item_description = []

        for row in rows:
            linha_dividida = row.split(" ")

            for informacao_linha in linha_dividida: 
                informacao_limpa = re.sub(r"[^a-zA-Z]", "", informacao_linha)
                informacao_limpa_com_numero = re.sub(r"[^a-zA-Z0-9]", "", informacao_linha)           
                if informacao_limpa.isalpha() and not informacao_limpa_com_numero[-1].isdigit(): #Valida se é apenas caractere do alfabeto
                    current_item_description.append(informacao_linha)
                    print(current_item_description)
                elif informacao_linha[-1].isdigit() and "," not in informacao_linha and informacao_linha in unidades_medida: #Valida se são apenas valores
                    qtd.append(informacao_linha[-1])
                    print(qtd)
                elif "," in informacao_linha:
                    current_item.append(informacao_linha)
                    print(current_item)


        # for row in rows:
        #     print(row)
        #     parts = row.split()
        #     if len(parts) > 6 and parts[-2].replace(',', '').isdigit() and parts[-1].isalpha():
        #         if current_item:
        #             # Combine description and append to current_item
        #             current_item.append(' '.join(current_item_description))
        #             items.append(current_item)
        #             current_item = []
        #             current_item_description = []
        #         if len(parts) >= 8:
        #             current_item = [parts[-8], parts[-7], parts[-6], parts[-5] + ' ' + parts[-4], parts[-3], parts[-2], parts[-1]]
        #             current_item_description = parts[:-8]
        #     else:
        #         current_item_description += parts

        # if current_item:
        #     current_item.append(' '.join(current_item_description))
        #     items.append(current_item)

        # # Criar DataFrame para os itens
        # data = []
        # for item in items:
        #     try:
        #         vlr_total = item[0]
        #         codigo = item[1]
        #         qtd = item[2]
        #         desc = item[3]
        #         vlr_unit = item[4]
        #         cod_barras = item[5]
        #         fabricante = item[6]
        #         descricao = ' '.join(item[7:])
        #         data.append([vlr_total, codigo, qtd, desc, vlr_unit, cod_barras, fabricante, descricao])
        #     except IndexError:
        #         continue

        # columns = ["Valor Total", "Código", "Qtd.", "Desc.%", "Vlr Unit", "Cod. Barras", "Fabricante", "Descrição"]
        # df_items = pd.DataFrame(data, columns=columns)

        # cotacoes.append({
        #     "numero_pedido": numero_pedido,
        #     "cnpj_filial": cnpj_filial,
        #     "data_emissao": data_emissao,
        #     "itens": df_items
        # })

    return cotacoes

file_path = r"C:\Users\Nicolas\Downloads\PDF\PEDIDO KOLESTON.pdf"
cotacoes = extract_pdf_data(file_path)

# Print the extracted data
for cotacao in cotacoes:
    print(f'Número do Pedido: {cotacao["numero_pedido"]}')
    print(f'CNPJ da Filial: {cotacao["cnpj_filial"]}')
    print(f'Data de Emissão do Pedido: {cotacao["data_emissao"]}')
    print('Tabela de Itens:')
    print(cotacao["itens"])
    print('\n' + '='*80 + '\n')

AttributeError: 'NoneType' object has no attribute 'group'